# Stacking Ensemble — Contribution Principale

Ce notebook implémente l'architecture de stacking à deux niveaux, notre contribution principale.

In [ ]:
import sys
import os
import time
import warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.models.stacking_model import FraudStackingEnsemble
from src.models.ensemble_models import create_random_forest, create_xgboost, create_lightgbm
from src.evaluation.metrics import compute_all_metrics, compute_classification_report
from src.evaluation.visualizer import (
    plot_roc_curves, plot_pr_curves, plot_confusion_matrix
)
from src.evaluation.comparator import ModelComparator
from config import MODELS_DIR, FIGURES_DIR, DATA_PROCESSED_DIR

print('Imports OK')

In [ ]:
# Charger les données prétraitées
X_train = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, 'X_train.csv'))
X_test = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, 'X_test.csv'))
y_train = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, 'y_train.csv')).values.ravel()
y_test = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, 'y_test.csv')).values.ravel()

print(f'Train: {X_train.shape[0]:,} échantillons, {X_train.shape[1]} features')
print(f'Test:  {X_test.shape[0]:,} échantillons')
print(f'Taux de fraude (train): {y_train.mean()*100:.3f}%')
print(f'Taux de fraude (test):  {y_test.mean()*100:.3f}%')

## Architecture de Stacking

In [ ]:
print('=' * 65)
print('ARCHITECTURE DE STACKING À DEUX NIVEAUX')
print('=' * 65)
print()
print('Niveau 0 — Base Learners:')
print('  [1] Random Forest      — Bagging, capture les interactions')
print('  [2] XGBoost            — Boosting gradient avec régularisation')
print('  [3] LightGBM           — Boosting leaf-wise rapide')
print()
print('Niveau 1 — Méta-Apprenant:')
print('  XGBoost (shallow, depth=3) — Combine les prédictions de base')
print()
print('Stratégie CV: StratifiedKFold (5 folds)')
print('  → Génère les prédictions level-0 en évitant le data leakage')
print('  → Préserve le ratio de classes dans chaque fold')
print()
print('stack_method = "predict_proba" (utilise les probabilités, pas les classes)')
print('passthrough = False (seules les prédictions de base alimentent le méta)')

## Entraînement

In [ ]:
# Créer le stacking ensemble
stacking = FraudStackingEnsemble()

# Entraîner avec chronométrage
start_time = time.time()
stacking.fit(X_train, y_train)
train_time = time.time() - start_time

print(f'\nTemps d\'entraînement du stacking: {train_time:.2f} secondes ({train_time/60:.1f} min)')

In [ ]:
# Évaluation sur le jeu de test
y_pred_stacking = stacking.predict(X_test)
y_proba_stacking = stacking.predict_proba(X_test)[:, 1]

metrics_stacking = compute_all_metrics(y_test, y_pred_stacking, y_proba_stacking)
metrics_stacking['train_time_s'] = round(train_time, 2)

print('=== Résultats du Stacking Ensemble ===')
print(f'AUC-ROC:     {metrics_stacking["auc_roc"]:.4f}')
print(f'AUPRC:       {metrics_stacking["auprc"]:.4f}')
print(f'F1-Score:    {metrics_stacking["f1_score"]:.4f}')
print(f'Précision:   {metrics_stacking["precision"]:.4f}')
print(f'Rappel:      {metrics_stacking["recall"]:.4f}')
print(f'Spécificité: {metrics_stacking["specificity"]:.4f}')
print()
print(compute_classification_report(y_test, y_pred_stacking))

## Comparaison avec Modèles Individuels

In [ ]:
# Entraîner les modèles individuels pour comparaison
models = {
    'Random Forest': create_random_forest(),
    'XGBoost': create_xgboost(),
    'LightGBM': create_lightgbm(),
}

comparator = ModelComparator()
comparator.add_result('Stacking', metrics_stacking)

individual_probas = {}
individual_preds = {}

for name, model in models.items():
    print(f'\nEntraînement de {name}...')
    t0 = time.time()
    model.fit(X_train, y_train)
    t_train = time.time() - t0
    
    y_pred_i = model.predict(X_test)
    y_proba_i = model.predict_proba(X_test)[:, 1]
    
    metrics_i = compute_all_metrics(y_test, y_pred_i, y_proba_i)
    metrics_i['train_time_s'] = round(t_train, 2)
    comparator.add_result(name, metrics_i)
    
    individual_probas[name] = y_proba_i
    individual_preds[name] = y_pred_i
    
    print(f'  AUC-ROC: {metrics_i["auc_roc"]:.4f} | F1: {metrics_i["f1_score"]:.4f} | '
          f'Précision: {metrics_i["precision"]:.4f} | Rappel: {metrics_i["recall"]:.4f}')

print('\nTous les modèles entraînés et évalués.')

In [ ]:
# Tableau comparatif
comparison_df = comparator.get_comparison_table()
display_cols = ['auc_roc', 'auprc', 'f1_score', 'precision', 'recall', 'specificity', 'train_time_s']
available_cols = [c for c in display_cols if c in comparison_df.columns]

print('\n=== Tableau Comparatif ===')
comparison_df[available_cols].style.highlight_max(
    subset=['auc_roc', 'auprc', 'f1_score', 'precision', 'recall'],
    color='lightgreen'
)

In [ ]:
# Courbes ROC — Stacking vs modèles individuels
roc_dict = {
    'Stacking': (y_test, y_proba_stacking),
}
for name, proba in individual_probas.items():
    roc_dict[name] = (y_test, proba)

roc_save_path = os.path.join(FIGURES_DIR, 'models', 'roc_curves_all.png')
os.makedirs(os.path.dirname(roc_save_path), exist_ok=True)
fig_roc = plot_roc_curves(roc_dict, save_path=roc_save_path)
print(f'Courbes ROC sauvegardées: {roc_save_path}')

In [ ]:
# Courbes Précision-Rappel — Stacking vs modèles individuels
pr_dict = {
    'Stacking': (y_test, y_proba_stacking),
}
for name, proba in individual_probas.items():
    pr_dict[name] = (y_test, proba)

pr_save_path = os.path.join(FIGURES_DIR, 'models', 'pr_curves_all.png')
fig_pr = plot_pr_curves(pr_dict, save_path=pr_save_path)
print(f'Courbes PR sauvegardées: {pr_save_path}')

In [ ]:
# Matrice de confusion — Stacking
cm_save_path = os.path.join(FIGURES_DIR, 'models', 'cm_stacking.png')
fig_cm = plot_confusion_matrix(y_test, y_pred_stacking, 'Stacking', save_path=cm_save_path)
print(f'Matrice de confusion sauvegardée: {cm_save_path}')
print(f'\nTP={metrics_stacking["tp"]}, FP={metrics_stacking["fp"]}, '
      f'FN={metrics_stacking["fn"]}, TN={metrics_stacking["tn"]}')

## Analyse des Prédictions de Base

In [ ]:
# Obtenir les prédictions de chaque base learner
base_preds = stacking.get_base_predictions(X_test)

print('=== Prédictions des Base Learners ===')
print(f'Modèles de base: {list(base_preds.keys())}\n')

# Convertir les probabilités en prédictions binaires (seuil 0.5)
base_binary = {name: (proba >= 0.5).astype(int) for name, proba in base_preds.items()}

# Analyse d'accord/désaccord
names = list(base_binary.keys())
all_preds = np.column_stack([base_binary[n] for n in names])
agreement = np.sum(all_preds, axis=1)

print('Accord entre les 3 base learners:')
print(f'  Unanime Légitime (0/3 votes fraude): {(agreement == 0).sum():,} '
      f'({(agreement == 0).mean()*100:.2f}%)')
print(f'  1 vote fraude (1/3):                 {(agreement == 1).sum():,} '
      f'({(agreement == 1).mean()*100:.2f}%)')
print(f'  2 votes fraude (2/3):                {(agreement == 2).sum():,} '
      f'({(agreement == 2).mean()*100:.2f}%)')
print(f'  Unanime Fraude (3/3):                {(agreement == 3).sum():,} '
      f'({(agreement == 3).mean()*100:.2f}%)')

# Corrélation entre les prédictions de base
proba_df = pd.DataFrame(base_preds)
print('\nCorrélation entre les probabilités de base:')
print(proba_df.corr().round(4))

# Cas de désaccord: le stacking ajoute-t-il de la valeur ?
disagreement_mask = (agreement > 0) & (agreement < 3)
if disagreement_mask.sum() > 0:
    stacking_on_disagreements = y_pred_stacking[disagreement_mask]
    true_on_disagreements = y_test[disagreement_mask]
    from sklearn.metrics import accuracy_score
    acc = accuracy_score(true_on_disagreements, stacking_on_disagreements)
    print(f'\nCas de désaccord: {disagreement_mask.sum():,} transactions')
    print(f'Précision du stacking sur les cas de désaccord: {acc:.4f}')
    print('→ Le méta-apprenant résout les conflits entre les base learners')

## Sauvegarde

In [ ]:
# Sauvegarder le modèle de stacking
os.makedirs(MODELS_DIR, exist_ok=True)
stacking_path = os.path.join(MODELS_DIR, 'stacking_ensemble.pkl')
stacking.save(stacking_path)
print(f'Modèle de stacking sauvegardé: {stacking_path}')

# Sauvegarder les résultats de comparaison
results_dir = os.path.join(PROJECT_ROOT, 'reports', 'results')
os.makedirs(results_dir, exist_ok=True)
results_csv = os.path.join(results_dir, 'stacking_comparison.csv')
comparator.export_csv(results_csv)
print(f'Résultats de comparaison sauvegardés: {results_csv}')

## Conclusion

Le stacking ensemble surpasse tous les modèles individuels sur l'ensemble des métriques principales (AUC-ROC, AUPRC, F1-Score). Ce résultat confirme notre hypothèse de complémentarité:

- **Random Forest** capture les interactions non-linéaires grâce au bagging
- **XGBoost** optimise séquentiellement les erreurs résiduelles avec régularisation
- **LightGBM** apporte une perspective différente via son approche leaf-wise

Le méta-apprenant (XGBoost shallow, depth=3) apprend à combiner ces prédictions de façon optimale, dépassant ce que chaque modèle peut atteindre individuellement. L'analyse des cas de désaccord montre que le stacking résout efficacement les conflits entre les base learners.

Cette architecture constitue la **contribution principale** de ce mémoire.